<a href="https://colab.research.google.com/github/tristanengst/fintechCS9832025fa/blob/main/5FinGPT_Forecaster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CASE5 FinGPT_Forecaster**


**Purpose**:
This section demonstrates how to use the FinGPT forecasting pipeline to predict stock price movements using historical price and sentiment data. It is adapted from the official FinGPT forecasting demo notebook.



---
## Instructions

**Step 1 – Switch to A100 runtime (Colab only)**  
> Purpose: loading FinGPT v3.2 (LLaMA2-based) requires a powerful GPU italicized text, such as A100 or RTX 3090.  

1. Go to **Runtime → Change runtime type** in the top menu.  
2. Set **Hardware accelerator** to `A100 GPU`.  
3. To access A100 in Google Colab, you may need to:  
   - Upgrade to *Colab Pro*, or  
   - Use *Colab Pay-As-You-Go* (**recommended for flexibility**)  
   - [Click here](https://colab.research.google.com/signup) to upgrade or enable Pay-As-You-Go.  
4. Click **Save**.

**Step 2 - Access Requirement for LLaMA-2-7B-Chat-HF Model**  
  This notebook uses the meta-llama/Llama-2-7b-chat-hf model from Hugging Face. Same as Notebook 4 on Multi-Task Financial LLMs.
  To run this code, you must request access from Meta and Hugging Face:
  1. Go to the model page: https://huggingface.co/meta-llama/Llama-2-7b-chat-hf
  2. Click “Expand to review and access”
  3. Fill in your contact information as required and submit the request.
  4. Wait for the approval email from HuggingFace. It usually takes around one business day. The email subject will be: [Access granted] Your request to access model meta-llama/Llama-2-70b-hf has been accepted.
  5. Your HF_token on HuggingFace should have Permissions set to "finegrained". You also need to go into "edit permissions" and enable read access to repositories.

Once your request is approved, you can proceed to the next step in this notebook.


**Step 3 – Set your API key**  
 Please set your API keys as environment variables to avoid exposing sensitive information in the code.

  On the left sidebar, click the “🔑 Secrets” icon.

  Add the following secret:

1. HF_TOKEN
 - HF_TOKEN refers to your Hugging Face access token.
 - Visit: https://huggingface.co/settings/tokens

Enable "Notebook Access" on the left side of each secret.

**Step 4 – Run the entire notebook**  
Once the dependencies are installed and runtime is set, go to **Runtime → Run all** to execute all cells.

---

## Code

### Environment Setup and Dependency Installation

In [2]:
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu126
!pip install transformers==4.32.0 peft==0.5.0
!pip install -U bitsandbytes>=0.46.1
!pip uninstall -y torchao

# check for GPU availability
import torch
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("CUDA device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

Looking in indexes: https://download.pytorch.org/whl/cu126
  Using cached transformers-4.32.0-py3-none-any.whl.metadata (118 kB)
  Using cached peft-0.5.0-py3-none-any.whl.metadata (22 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.13.3.tar.gz (314 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using cached transformers-4.32.0-py3-none-any.whl (7.5 MB)
Using cached peft-0.5.0-py3-none-any.whl (85 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
error: failed-wheel-build-for-install

× Failed to build ins

In [3]:
!git clone https://github.com/AI4Finance-Foundation/FinGPT.git
%cd FinGPT/fingpt/FinGPT_Forecaster/
!pip install peft pandas yfinance finnhub-python
!pip install rouge-score
#!pip uninstall -y safetensors
#!pip install safetensors==0.3.1

Cloning into 'FinGPT'...
remote: Enumerating objects: 3298, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 3298 (delta 15), reused 14 (delta 11), pack-reused 3262 (from 3)
Receiving objects: 100% (3298/3298), 12.90 MiB | 21.24 MiB/s, done.
Resolving deltas: 100% (1845/1845), done.
/content/FinGPT/fingpt/FinGPT_Forecaster
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24985 sha256=d0a747ff7460e2b981fc2beb10af67206dfa25bd2ccc3b712e3dc885f43b15fa
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge-score


In [4]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from datasets import load_from_disk
# from peft import PeftModel
from utils import *

os.environ['CUDA_VISIBLE_DEVICES'] = "5,6"

### Load Base Model and Fine-Tuned Weights for Forecasting

In [5]:
base_model = AutoModelForCausalLM.from_pretrained(
    'meta-llama/Llama-2-7b-chat-hf',
    trust_remote_code=True,
    device_map="cuda:0",
    torch_dtype=torch.float16,
)
base_model.model_parellal = True
# I'm getting an error accessing HF. OS Error: You are trying to access a gated repo.
# Make sure to request access at https://huggingface.co/meta-llama/Llama-2-7b-chat-hf and pass a token having permission to this repo
# either by logging in with `huggingface-cli login` or by passing `token=<your_token>`.

config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

In [6]:
from peft import PeftModel

model = PeftModel.from_pretrained(base_model, 'FinGPT/fingpt-forecaster_dow30_llama2-7b_lora')
model = model.eval()

adapter_config.json:   0%|          | 0.00/528 [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 40.0MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

In [8]:
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Llama-2-7b-chat-hf')

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

### Generate Stock Price Forecast Using Prompt Engineering and FinGPT Model

In [9]:
import re

SYSTEM_PROMPT = "You are a seasoned stock market analyst. Your task is to list the positive developments and potential concerns for companies based on relevant news and basic financials from the past weeks, then provide an analysis and prediction for the companies' stock price movement for the upcoming week. Your answer format should be as follows:\n\n[Positive Developments]:\n1. ...\n\n[Potential Concerns]:\n1. ...\n\n[Prediction & Analysis]:\n..."

prompt = """
[Company Introduction]:

{name} is a leading entity in the {finnhubIndustry} sector. Incorporated and publicly traded since {ipo}, the company has established its reputation as one of the key players in the market. As of today, {name} has a market capitalization of {marketCapitalization:.2f} in {currency}, with {shareOutstanding:.2f} shares outstanding. {name} operates primarily in the {country}, trading under the ticker {ticker} on the {exchange}. As a dominant force in the {finnhubIndustry} space, the company continues to innovate and drive progress within the industry.

From {startDate} to {endDate}, {name}'s stock price increased from {startPrice} to {endPrice}. Company news during this period are listed below:

[Headline]: Apple launches new Vision Pro headset
[Summary]: Apple announced the launch of its AR/VR headset Vision Pro, marking a significant step into mixed reality.

[Headline]: Record quarterly revenue
[Summary]: Apple reported record revenue of $123B for the last quarter, driven by strong iPhone sales.

Some recent basic financials of {name}, reported at {curday}, are presented below:

[Basic Financials]:
{attr1}: {value1}
{attr2}: {value2}

Based on all the information before {curday}, let's first analyze the positive developments and potential concerns for {symbol}. Come up with 2-4 most important factors respectively and keep them concise. Most factors should be inferred from company-related news. Then make your prediction of the {symbol} stock price movement for next week ({period}). Provide a summary analysis to support your prediction.
"""

data = {
    "name": "Apple Inc.",
    "finnhubIndustry": "Technology",
    "ipo": "1980-12-12",
    "marketCapitalization": 2900000000000,
    "currency": "USD",
    "shareOutstanding": 16000000000,
    "country": "United States",
    "ticker": "AAPL",
    "exchange": "NASDAQ",
    "startDate": "2023-06-01",
    "endDate": "2023-06-30",
    "startPrice": "$175",
    "endPrice": "$185",
    "symbol": "AAPL",
    "period": "2023-07-01 to 2023-07-07",
    "curday": "2023-06-30",
    "attr1": "Revenue (TTM)",
    "value1": "$385B",
    "attr2": "Net Income (TTM)",
    "value2": "$99B",
}

prompt_filled = prompt.format(**data)

B_INST, E_INST = "[INST]", "[/INST]"
B_SYS, E_SYS = "<<SYS>>\n", "\n<</SYS>>\n\n"

final_prompt = B_INST + B_SYS + SYSTEM_PROMPT + E_SYS + prompt_filled + E_INST

inputs = tokenizer(final_prompt, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

res = model.generate(
    **inputs,
    max_length=4096,
    do_sample=True,
    eos_token_id=tokenizer.eos_token_id,
    use_cache=True
)

output = tokenizer.decode(res[0], skip_special_tokens=True)
answer = re.sub(r'.*\[\/INST\]\s*', '', output, flags=re.DOTALL)

print(answer)


[Positive Developments]:
1. The launch of the new Vision Pro headset by Apple, a significant step into mixed reality, which could potentially increase the company's market share in the technology space.
2. Record quarterly revenue of $123B, driven by strong iPhone sales, indicating a healthy demand for the company's products.
3. The company's revenue and net income are both increasing, indicating financial stability and profitability.

[Potential Concerns]:
1. The high valuation of Apple's stock, which is reflected in its price-to-earnings ratio (P/E) of 16.47, could make it vulnerable to a correction.
2. The ongoing competition in the technology sector, particularly from Chinese tech giants like Huawei, could pose a threat to Apple's market share.
3. The launch of new products may not always lead to an increase in stock price, as it may depend on market perception and consumer demand.

[Prediction & Analysis]:
Based on the above analysis, I predict that the stock price of Apple Inc. w

## Declaration

Source repo: AI4Finance-Foundation/fingpt/[FinGPT_Forecaster](https://github.com/AI4Finance-Foundation/FinGPT/blob/master/fingpt/FinGPT_Forecaster/demo.ipynb)